[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/15_mlp_solution.ipynb)

# 🟠 Solution: SwiGLU MLP（参考版）

## 📋 题目描述

**难度：** Medium

实现 SwiGLU MLP（nn.Module）。

SwiGLU 是 LLaMA 等现代 LLM 使用的门控 MLP 变体：对一个投影应用 SiLU 门控，与另一个投影逐元素相乘。

**签名:** `SwiGLUMLP(d_model, d_ff)`（nn.Module）

**前向传播:** `forward(x) -> Tensor`
- `x` — 输入张量 (*, d_model)

**返回:** 输出张量 (*, d_model)

**约束:**
- 三个投影：gate_proj(d, d_ff)、up_proj(d, d_ff)、down_proj(d_ff, d)
- `forward(x) = down_proj(silu(gate_proj(x)) * up_proj(x))`
- SiLU(x) = x * sigmoid(x)

**提示：** `gate_proj(d→d_ff)`、`up_proj(d→d_ff)`、`down_proj(d_ff→d)`。前向：`down_proj(silu(gate_proj(x)) * up_proj(x))`。`silu(x) = x * sigmoid(x)`。


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ SOLUTION

class SwiGLUMLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # ---- Step 1: 定义三个线性投影 ----
        # gate_proj: d_model → d_ff，用于生成"门控信号"
        # up_proj:   d_model → d_ff，用于生成"候选激活值"
        # down_proj: d_ff → d_model，将高维特征压缩回原始维度
        # 为什么是三个而不是两个？SwiGLU 的核心思想是用一个投影做门控，
        # 另一个投影做值，两者逐元素相乘后再投影回来
        self.gate_proj = nn.Linear(d_model, d_ff)
        self.up_proj = nn.Linear(d_model, d_ff)
        self.down_proj = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # ---- Step 2: 计算门控 MLP ----
        # gate_proj(x) → (*, d_ff)：门控分支的原始值
        # F.silu(gate_proj(x)) → (*, d_ff)：SiLU 激活 = x * sigmoid(x)
        #   SiLU 是平滑的 ReLU 替代品，零点附近有负值（负值区域有小的非零输出）
        # up_proj(x) → (*, d_ff)：值分支，不做激活
        # 两者逐元素相乘：门控信号决定"让多少值通过"
        # down_proj(...) → (*, d_model)：投影回原始维度
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


In [ ]:
mlp = SwiGLUMLP(d_model=64, d_ff=128)
x = torch.randn(2, 8, 64)
print('Output:', mlp(x).shape)
print('Params:', sum(p.numel() for p in mlp.parameters()))


In [ ]:
from torch_judge import check
check('mlp')


## 📝 核心思路总结

1. **门控 MLP 结构**：gate_proj 生成门控信号，up_proj 生成候选值，逐元素相乘后 down_proj 降维
2. **SiLU 激活**：`x * sigmoid(x)`，平滑的 ReLU 替代品，零点附近有负值输出
3. **为什么 SwiGLU 更好**：门控机制让网络学习"选择性激活"，比单纯的非线性激活更灵活
4. **参数量权衡**：3 个投影 vs 标准 MLP 的 2 个，参数多 50%，但 LLaMA 等模型证明值得
